# Apache Spark en Google Colab
Ejercicios de WordCount, DataFrame API y MLlib (clasificación)

In [1]:
#configuración en google colab de spark y pyspark
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz
!tar xf spark-4.0.1-bin-hadoop3.tgz
!pip install -q findspark
!pip install -q pyspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-4.0.1-bin-hadoop3"
import findspark
findspark.init()

## Ejemplo 1: WordCount con RDD

In [3]:
from pyspark import SparkContext
sc = SparkContext.getOrCreate()

text = sc.textFile("gdrive/MyDrive/datasets/gutenberg-small/*.txt")
counts = text.flatMap(lambda x: x.split(" ")) \
             .map(lambda x: (x, 1)) \
             .reduceByKey(lambda a, b: a + b)
counts.collect()

[('', 27298),
 ('Published', 3),
 ('themselves', 192),
 ('were', 1450),
 ('sheet', 4),
 ('despatched', 4),
 ('most', 551),
 ('turbulent', 2),
 ('A.', 1456),
 ('ORIGINALS', 1),
 ('IN', 84),
 ('BROTHER]', 2),
 ('more', 1211),
 ('forget', 46),
 ('prove', 113),
 ('Give', 14),
 ('Johnston:--', 1),
 ('request', 53),
 ('comply', 10),
 ('times', 117),
 ('know.', 24),
 ('doubt', 134),
 ("day's", 8),
 ('does', 445),
 ('what', 1162),
 ('nail,"', 3),
 ('Let', 249),
 ('fair', 127),
 ('own', 664),
 ('dollar.', 5),
 ('yourself', 67),
 ('County.', 16),
 ('ever.', 20),
 ("months'", 7),
 ('it?', 167),
 ('mine.', 24),
 ('power', 430),
 ('give,', 10),
 ('cause,', 44),
 ('APPROVAL', 2),
 ('First', 28),
 ('States,', 567),
 ('consider', 165),
 ('seems', 147),
 ('accession', 8),
 ('Republican', 186),
 ('cause', 184),
 ('believe', 421),
 ('so,', 180),
 ('Those', 38),
 ('acceptance,', 7),
 ('rights', 124),
 ('control', 125),
 ('invasion', 23),
 ('under', 741),
 ('pretext,', 4),
 ('sentiments;', 6),
 ('susceptib

## Ejemplo 2: Análisis con DataFrame API

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Simular DataFrame de ventas
data = [("martillo", 12000), ("taladro", 45000), ("martillo", 15000)]
columns = ["producto", "valor"]
df = spark.createDataFrame(data, columns)
df.groupBy("producto").sum("valor").show()

+--------+----------+
|producto|sum(valor)|
+--------+----------+
|martillo|     27000|
| taladro|     45000|
+--------+----------+



## Ejemplo 3: Clasificación con MLlib

In [5]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

df = spark.read.csv("gdrive/MyDrive/datasets/clientes.csv", header=True, inferSchema=True)

assembler = VectorAssembler(inputCols=["edad", "ingresos"], outputCol="features")
data = assembler.transform(df).select("features", df["comprador"].alias("label"))
train, test = data.randomSplit([0.8, 0.2], seed=42)
lr = LogisticRegression()
model = lr.fit(train)
model.transform(test).select("features", "label", "prediction").show()

+-------------+-----+----------+
|     features|label|prediction|
+-------------+-----+----------+
|[34.0,4500.0]|    1|       0.0|
+-------------+-----+----------+



## Ejemplo 4: Spark GraphX

In [31]:
!pip install -q pyspark
!pip install -q graphframes

!wget -q https://repo1.maven.org/maven2/graphframes/graphframes/0.8.3-spark3.5-s_2.12/graphframes-0.8.3-spark3.5-s_2.12.jar -P /content/jars

In [32]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--jars /content/jars/graphframes-0.8.3-spark3.5-s_2.12.jar pyspark-shell"
)

In [33]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GraphFrames PageRank") \
    .getOrCreate()

In [36]:
from graphframes import GraphFrame

# DataFrame de vértices
vertices = spark.createDataFrame([("1", "A"), ("2", "B"), ("3", "C"), ("4", "D") ], ["id", "name"])

# DataFrame de edges
edges = spark.createDataFrame([("1", "2"),    ("2", "3"),    ("3", "4"),    ("4", "1")], ["src", "dst"])

# grafo
# g = GraphFrame(vertices, edges)

## Error de GraphFrames

Según explicó ChatGPT:

> GraphFrames does not load in Colab’s Spark 4.x runtime because GraphFrames’ latest release (0.8.3) is built for Spark 3.5, while Google Colab now bundles Spark 4.0.1.
>
> The JVM classloader rejects the JAR, leaving org.graphframes.GraphFramePythonAPI unavailable, which produces the exact error you see.
>
> There is no compatible GraphFrames build for Spark 4.x.
>
> To use GraphFrames in Colab, you must run Spark 3.5.x, not Spark 4.x.

In [37]:
# algoritmo de PageRank
# results = g.pageRank(resetProbability=0.15, maxIter=10)

# resultados de PageRank
# results.vertices.select("id", "name", "pagerank").show()